# 01 · Data exploration — PhysioNet/CinC Challenge 2017

**Goal**: sanity-check the dataset before any modelling.

**Questions answered here**:
1. How many records per class? (class imbalance is the first failure-mode driver)
2. What does a *typical* record look like for each rhythm class?
3. How long are the recordings — does our 30-s window pad or crop most of them?
4. Are there obvious quality issues (flat lines, saturation, missing leads)?

Run this first; everything downstream assumes the dataset behaves as documented.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

from af_explain.data.dataset import LABEL_MAP, LABEL_NAMES
from af_explain.data.preprocess import DEFAULT_FS, preprocess_record

DATA_ROOT = Path('../data/raw/training2017')
assert DATA_ROOT.exists(), 'Run `./scripts/download_data.sh` first.'

## 1. Class balance

In [ ]:
refs = pd.read_csv(DATA_ROOT / 'REFERENCE.csv', header=None, names=['record', 'label'])
refs['rhythm'] = refs['label'].map({k: v for k, v in zip(LABEL_MAP.keys(), LABEL_NAMES, strict=True)})
counts = refs['rhythm'].value_counts().reindex(LABEL_NAMES)
counts.plot(kind='bar', color=['#2ca02c', '#d62728', '#1f77b4', '#7f7f7f'])
plt.ylabel('records')
plt.title('PhysioNet/CinC 2017 — class balance')
plt.tight_layout()
counts

## 2. One example per class (raw + preprocessed)

Use this to build intuition for *what the model sees* before any training.

In [ ]:
samples = (
    refs.groupby('rhythm', group_keys=False).apply(lambda g: g.sample(1, random_state=0))
).reset_index(drop=True)

fig, axes = plt.subplots(len(samples), 2, figsize=(14, 2.5 * len(samples)), sharex=True)
for i, row in samples.iterrows():
    sig, _ = wfdb.rdsamp(str(DATA_ROOT / row['record']))
    raw = sig[:, 0].astype(np.float32)
    proc, _ = preprocess_record(raw, fs=DEFAULT_FS)
    t_raw = np.arange(len(raw)) / DEFAULT_FS
    t_proc = np.arange(len(proc)) / DEFAULT_FS
    axes[i, 0].plot(t_raw, raw, lw=0.6)
    axes[i, 0].set_title(f"{row['rhythm']}  ·  {row['record']}  ·  raw")
    axes[i, 1].plot(t_proc, proc, lw=0.6, color='black')
    axes[i, 1].set_title('preprocessed (bandpass + z-score + 30 s)')
    axes[i, 0].set_ylabel('amplitude')
axes[-1, 0].set_xlabel('seconds')
axes[-1, 1].set_xlabel('seconds')
plt.tight_layout()

## 3. Recording length distribution

Tells us how much we pad vs crop with the 30-s window.

In [ ]:
def record_length_seconds(record_id: str) -> float:
    header = wfdb.rdheader(str(DATA_ROOT / record_id))
    return header.sig_len / header.fs

lengths = refs.sample(min(len(refs), 1500), random_state=0)['record'].map(record_length_seconds)
lengths.plot(kind='hist', bins=40)
plt.axvline(30, color='red', ls='--', label='30 s window')
plt.xlabel('seconds')
plt.legend()
plt.title('Record length distribution (sample of 1500)')
plt.tight_layout()

## TODO — your turn

- Inspect 2-3 noisy records (`label == '~'`). Do you spot the artifact yourself?
- For one AFib record, mark by eye the windows where you'd point a colleague to support the diagnosis. These annotations will feed `clinician_concordance` later.